# Thermodynamic Analysis of Unidentified Photosynthetic Modes

This notebook investigates flux solutions classified as "Unidentified" (neither C3, C4, nor intermediate types) to determine their thermodynamic feasibility. 

**Workflow:**
1.  **Data Loading & Preprocessing:** Load the metabolic model and flux scan results.
2.  **Flux Normalization:** Normalize fluxes based on anatomical constraints (volume ratios).
3.  **Classification:** Filter for flux solutions that do not fit standard photosynthetic definitions.
4.  **Reaction Identification:** Identify reactions producing CO2 in these unidentified solutions.
5.  **Thermodynamic Analysis:** Use `equilibrator_api` to calculate $\Delta G'$ for candidate reactions.
6.  **Monte Carlo Sampling:** Assess thermodynamic feasibility under physiological concentration ranges.

## 1. Imports and Setup

In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
from cobra.io import read_sbml_model
import pandas as pd
from Flux import *
import numpy as np
import random
import re
import seaborn as sns
import matplotlib.pyplot as plt
import os
import math
from equilibrator_api import ComponentContribution, Q_, Reaction
import cvxpy as cp

## 2. Model and Data Loading
Load the C4 metabolic model and the dataset containing flux scan results.

In [2]:
c4_model = read_sbml_model("../Models/c4_model.xml")

Set parameter Username
Academic license - for non-commercial use only - expires 2027-01-20


In [3]:
df_Variable = pd.read_csv("../Datasets/Scan_Variable_Weighted.csv")

In [4]:
df_variable_structured = structure_data(df=df_Variable, model=c4_model)

In [5]:
df_variable_structured

,Vc_Vo,PPFD,V_BS_M,Maximum Nitrogen Uptake,Maximum Carbon Uptake,Carbon Uptake,Nitrogen Uptake,Light Mesophyll,Light Bundle Sheath,PC_M,...,Glutamine_syn_c_M,Glutamine_syn_m_M,Glutamine_syn_p_M,Glutamine_syn_c_B,Glutamine_syn_m_B,Glutamine_syn_p_B,Glycolate oxidase M,Phosphoglycolate phosphatase M,Glycolate oxidase B,Phosphoglycolate phosphatase B
0,50.0,1000,0.4,1000.0,20.0,20.0,3.7,145.285714,193.057143,1.0,...,0.0,0.000000,3.571429,0.0,0.0,0.514286,0.0,0.0,0.2,0.2
1,55.0,1000,0.4,1000.0,20.0,20.0,3.7,117.571429,249.828571,1.0,...,0.0,0.000000,3.571429,0.0,0.0,3.285714,0.2,0.2,19.4,19.4
2,60.0,1000,0.4,1000.0,20.0,20.0,3.7,255.142857,170.542857,1.0,...,0.0,0.000000,7.571429,0.0,0.0,2.514286,10.6,10.6,15.7,15.7
3,65.0,1000,0.4,1000.0,20.0,20.0,3.7,394.428571,32.914286,1.0,...,0.0,0.071429,10.285714,0.0,0.0,0.142857,18.0,18.0,0.0,0.0
4,70.0,1000,0.4,1000.0,20.0,20.0,3.7,356.500000,27.314286,1.0,...,0.0,0.071429,8.571429,0.0,0.0,0.257143,13.6,13.6,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,80.0,1000,2.2,1000.0,20.0,20.0,3.7,97.812500,180.193750,1.0,...,0.0,0.000000,1.437500,0.0,0.0,2.612500,0.3,0.3,0.6,0.6
205,85.0,1000,2.2,1000.0,20.0,20.0,3.7,84.750000,205.631250,1.0,...,0.0,0.000000,0.875000,0.0,0.0,3.850000,2.1,2.1,2.6,2.6
206,90.0,1000,2.2,1000.0,20.0,20.0,3.7,35.000000,245.575000,1.0,...,0.0,0.000000,0.312500,0.0,0.0,4.262500,0.7,0.7,2.7,2.7
207,95.0,1000,2.2,1000.0,20.0,20.0,3.7,34.468750,231.893750,1.0,...,0.0,0.000000,0.500000,0.0,0.0,3.506250,0.4,0.4,1.2,1.2


## 3. Flux Normalization
Normalize flux values based on the volume ratio ($V_{BS}/V_{M}$) encoded in the column headers. This accounts for the anatomical differences between Bundle Sheath (BS) and Mesophyll (M) cells.

In [6]:
df = df_Variable

reaction_col = df.columns[0]
reactions = df[reaction_col].astype(str)

flux_cols = df.columns.tolist()[1:]

# Function to extract the volume ratio (v_bs_m) from a column name
def extract_volume_ratio(col_name):
    # Expecting something like "1500_0.1_1_1_50.0_20_1000" -> take second field after first underscore
    parts = str(col_name).split('_')
    try:
        return float(parts[1])
    except Exception:
        # fallback: regex search for a number after first underscore
        m = re.search(r'^[^_]*_(\d+(\.\d+)?)', str(col_name))
        return float(m.group(1)) if m else None

# Build a mapping of column -> volume_ratio and normalized volumes
col_info = []
for col in flux_cols:
    vr = extract_volume_ratio(col)
    if vr is None:
        vm_norm = None
        vbs_norm = None
    else:
        # Calculate normalization factors based on volume ratio
        vm_norm = 1.0 / (1.0 + vr)
        vbs_norm = vr / (1.0 + vr)
    col_info.append({
        "column": col, 
        "volume_ratio": vr, 
        "VM_Norm": vm_norm, 
        "VBS_Norm": vbs_norm
    })

col_info_df = pd.DataFrame(col_info)

# Pre-identify reaction types
is_mes = reactions.str.startswith("[M]")
is_bs = reactions.str.startswith("[B]")

# Check if reaction ends with "_tx" but is NOT a Photon_tx reaction
is_tx_non_photon = (
    reactions.str.endswith("_tx") & 
    ~reactions.isin(["[M]_Photon_tx", "[B]_Photon_tx"])
)

# Process all flux columns at once - build list of Series
adjusted_cols = []

for info in col_info:
    col = info["column"]
    vm = info["VM_Norm"]
    vbs = info["VBS_Norm"]

    # Copy original column as numeric (coerce errors to NaN)
    vals = pd.to_numeric(df[col], errors="coerce")

    # Determine if we should skip volume scaling for this reaction-column combination
    # Skip if: reaction ends with "_tx" AND is not a Photon_tx reaction
    skip_scaling = is_tx_non_photon
    
    # Apply scaling
    scaled = vals.copy()
    if vm is not None and vbs is not None:
        # Only scale reactions that should be scaled (non-tx or Photon_tx reactions)
        scale_mask_mes = is_mes & ~skip_scaling
        scale_mask_bs = is_bs & ~skip_scaling
        
        scaled.loc[scale_mask_mes] = vals.loc[scale_mask_mes] * vm
        scaled.loc[scale_mask_bs] = vals.loc[scale_mask_bs] * vbs
    
    adjusted_cols.append(scaled)

# Combine all columns at once using pd.concat (avoids fragmentation)
adjusted_df = pd.concat(
    [df[[reaction_col]]] + adjusted_cols,
    axis=1,
    keys=[reaction_col] + flux_cols
)

# Reset column names to match original structure
adjusted_df.columns = [reaction_col] + flux_cols

print("Shape:", adjusted_df.shape)
print("Number of flux columns:", len(flux_cols))

Shape: (1991, 210)
Number of flux columns: 209


## 4. Classification of Photosynthetic Modes
Filter the dataset to isolate flux solutions that do not conform to standard C3, C4, or intermediate definitions.

In [7]:
df_unidentified = df_variable_structured[ df_variable_structured["Photosynthesis"] == "Unidentified" ]
df_all_identified = df_variable_structured[ df_variable_structured["Photosynthesis"] != "Unidentified" ]

In [8]:
df_Variable.index = df_Variable["Unnamed: 0"]
df_Variable.drop(columns=["Unnamed: 0"], inplace=True)

In [9]:
def filter_unidentified_flux_solutions(df):
    """
    Filters flux solutions to only keep those classified as 'Unidentified'
    
    Parameters:
    df (pd.DataFrame): Original dataset with flux solutions as columns
    
    Returns:
    tuple: (pd.DataFrame containing 'Unidentified', pd.DataFrame containing 'Identified')
    """
    
    def photosynthetic_classification_single_column(column_data, column_name):
        """
        Classify a single flux solution column based on key enzymatic activities.
        """
        # Primary filter (10%) and dedicated lower threshold for GCV (5%)
        CO2_filter = float(column_name.split("_")[5]) * 0.1
        GCV_filter = float(column_name.split("_")[5]) * 0.05
        
        VM = 1 / (1 + float(column_name.split("_")[1]))
        VBS = float(column_name.split("_")[1]) / (1 + float(column_name.split("_")[1]))
        
        # Apply filtering to this column (zero out negligible fluxes)
        filtered_column = column_data.copy()
        for line in filtered_column.index:
            # Check GCV specifically with dedicated GCV threshold (volume normalized by VBS)
            if "GCVMULTI_RXN_m" in line:
                if -abs(GCV_filter) <= filtered_column.loc[line] * VBS <= GCV_filter:
                    filtered_column.loc[line] = 0
            elif "[B]" in line:
                if -abs(CO2_filter) <= filtered_column.loc[line] * VBS <= CO2_filter:
                    filtered_column.loc[line] = 0
            elif "[M]" in line:
                if -abs(CO2_filter) <= filtered_column.loc[line] * VM <= CO2_filter:
                    filtered_column.loc[line] = 0
        
        # Extract fluxes for key marker reactions
        M_rubisco = filtered_column.get("[M]_RIBULOSE_BISPHOSPHATE_CARBOXYLASE_RXN_p", 0)
        B_rubisco = filtered_column.get("[B]_RIBULOSE_BISPHOSPHATE_CARBOXYLASE_RXN_p", 0)
        B_rubisco_ex = filtered_column.get("[B]_RIBULOSE_BISPHOSPHATE_CARBOXYLASE_RXN_p_Ex", 0)
        B_pepck = filtered_column.get("[B]_PEPCARBOXYKIN_RXN_c", 0)
        B_malic_nadp = filtered_column.get("[B]_MALIC_NADP_RXN_p", 0)
        B_139_rxn = filtered_column.get("[B]_1_PERIOD_1_PERIOD_1_PERIOD_39_RXN_m", 0)
        M_pepcarbox = filtered_column.get("[M]_PEPCARBOX_RXN_c", 0)
        B_gcvmulti = filtered_column.get("[B]_GCVMULTI_RXN_m", 0)
        
        # Classification logic (Type II GCV constraint strictly maintained)
        if (M_rubisco > B_rubisco_ex ) and B_rubisco == 0 and B_pepck == 0 and B_malic_nadp == 0 and B_139_rxn == 0 and B_gcvmulti == 0:
            return "C3"
        elif M_rubisco != 0 and B_rubisco != 0 and B_gcvmulti != 0 and B_pepck == 0 and B_malic_nadp == 0 and B_139_rxn == 0:
            return "C3_C4_Type_I"
        elif M_rubisco != 0 and M_pepcarbox != 0 and B_rubisco != 0 and (B_pepck != 0 or B_malic_nadp != 0 or B_139_rxn != 0) and B_gcvmulti != 0:
            return "C3_C4_Type_II"
        elif M_rubisco == 0 and M_pepcarbox != 0 and B_rubisco != 0 and (B_pepck != 0 or B_malic_nadp != 0 or B_139_rxn != 0):
            return "C4"
        else:
            return "Unidentified"
    
    # Apply classification to each column and filter
    unidentified_columns = []
    identified_columns = []
    
    for column in df.columns:
        classification = photosynthetic_classification_single_column(df[column], column)
        if classification == "Unidentified":
            unidentified_columns.append(column)
        else:
            identified_columns.append(column)
    
    # Return only the unidentified columns (keeps original tuple return type)
    return df[unidentified_columns], df[identified_columns]


# Alternative function to process columns one by one
def is_unidentified_flux_solution(column_data, column_name):
    """
    Check if a single flux solution column is classified as 'Unidentified'
    
    Parameters:
    column_data (pd.Series): Single column of flux data
    column_name (str): Name of the column
    
    Returns:
    bool: True if classified as 'Unidentified', False otherwise
    """
    
    # Primary filter (10%) and dedicated lower threshold for GCV (0.5%)
    CO2_filter = float(column_name.split("_")[5]) * 0.1
    GCV_filter = float(column_name.split("_")[5]) * 0.05
    
    VM = 1 / (1 + float(column_name.split("_")[1]))
    VBS = float(column_name.split("_")[1]) / (1 + float(column_name.split("_")[1]))
    
    # Apply filtering
    filtered_column = column_data.copy()
    for line in filtered_column.index:
        if "GCVMULTI_RXN_m" in line:
            if -abs(GCV_filter) <= filtered_column.loc[line] * VBS <= GCV_filter:
                filtered_column.loc[line] = 0
        elif "[B]" in line:
            if -abs(CO2_filter) <= filtered_column.loc[line] * VBS <= CO2_filter:
                filtered_column.loc[line] = 0
        elif "[M]" in line:
            if -abs(CO2_filter) <= filtered_column.loc[line] * VM <= CO2_filter:
                filtered_column.loc[line] = 0
    
    # Extract fluxes for key marker reactions
    M_rubisco = filtered_column.get("[M]_RIBULOSE_BISPHOSPHATE_CARBOXYLASE_RXN_p", 0)
    B_rubisco = filtered_column.get("[B]_RIBULOSE_BISPHOSPHATE_CARBOXYLASE_RXN_p", 0)
    B_rubisco_ex = filtered_column.get("[B]_RIBULOSE_BISPHOSPHATE_CARBOXYLASE_RXN_p_Ex", 0)
    B_pepck = filtered_column.get("[B]_PEPCARBOXYKIN_RXN_c", 0)
    B_malic_nadp = filtered_column.get("[B]_MALIC_NADP_RXN_p", 0)
    B_139_rxn = filtered_column.get("[B]_1_PERIOD_1_PERIOD_1_PERIOD_39_RXN_m", 0)
    M_pepcarbox = filtered_column.get("[M]_PEPCARBOX_RXN_c", 0)
    B_gcvmulti = filtered_column.get("[B]_GCVMULTI_RXN_m", 0)
    
    # Classification logic
    if (M_rubisco > B_rubisco_ex ) and B_rubisco == 0 and B_pepck == 0 and B_malic_nadp == 0 and B_139_rxn == 0 and B_gcvmulti == 0:
        return False  # C3
    elif M_rubisco != 0 and B_rubisco != 0 and B_gcvmulti != 0 and B_pepck == 0 and B_malic_nadp == 0 and B_139_rxn == 0:
        return False  # C3_C4_Type_I
    elif M_rubisco != 0 and M_pepcarbox != 0 and B_rubisco != 0 and (B_pepck != 0 or B_malic_nadp != 0 or B_139_rxn != 0) and B_gcvmulti != 0:
        return False  # C3_C4_Type_II
    elif M_rubisco == 0 and M_pepcarbox != 0 and B_rubisco != 0 and (B_pepck != 0 or B_malic_nadp != 0 or B_139_rxn != 0):
        return False  # C4
    else:
        return True  # Unidentified

In [10]:
filtered_df, all_identified = filter_unidentified_flux_solutions(df_Variable)

## 5. Identification of CO2 Producing Reactions
Analyze the "Unidentified" solutions to find which reactions are producing CO2 in the Bundle Sheath, potentially indicating novel decarboxylation pathways.

In [11]:
def budget_plot(solution_frame, metabolite_query, cell_type):
    """
    Identifies producers and consumers of a specific metabolite in a given cell type.
    """
    #Defining the list of metabolites to search
    budget_metabolites = []

    #Defining the compartments within the model
    compartment = ["_c", "_m", "_p", "_x", "_e", "_i", "_v", "_l", "tx", "ss", "pc", "xc", "mc", "ep", 'mi']

    #Searching for the metabolites in the model
    for met in c4_model.metabolites.query(metabolite_query):
        if met.id[:3] == cell_type:
            budget_metabolites.append(met)

    #Defining list of reactions producing and consuming the metabolite
    consumers = []
    producers = []

    #Add reactions to respective list and exclude [B]_L_GLUTAMATE_GAMMA_SEMIALDEHYDE_m transport reactions
    for met in budget_metabolites:
        for reaction in c4_model.reactions:
            if met in reaction.reactants and reaction.id[-2:] in compartment:
                consumers.append(reaction.id)
            elif met in reaction.products and reaction.id[-2:] in compartment:
                producers.append(reaction.id)
    
    #Get flux values from the simulation for metabolite consuming/producing reactions
    producers_df = solution_frame.loc[producers,:]
    consumers_df = solution_frame.loc[consumers,:]

    #Get values with negative flows: producing reactions with negative flow are consuming and vice-versa
    negative_producers = list(producers_df[producers_df["fluxes"] < 0].index)
    negative_consumers = list(consumers_df[consumers_df["fluxes"] < 0].index)


    #Add reactions to correct list
    consumers.extend(negative_producers)
    producers.extend(negative_consumers)

    #Remove reactions with negative flux from old list
    def remove_items(test_list, item):
        res = [i for i in test_list if i != item]
        return res

    for item in negative_producers:
        producers = remove_items(producers, item)

    for item in negative_consumers:
        consumers = remove_items(consumers, item)

    #Get flux values from the simulation for metabolite consuming/producing reactions (correct list)
    producers_df = solution_frame.loc[producers,:]
    #Make all values positive (disregard directionality)

    producers_df["fluxes"] = producers_df["fluxes"].abs()
    #Remove reactions with zero flux
    producers_df = producers_df[(producers_df.T != 0).all()]

    #Get flux values from the simulation for metabolite consuming/producing reactions (correct list)
    consumers_df = solution_frame.loc[consumers,:]

    #Make all values positive (disregard directionality)
    consumers_df["fluxes"]  = consumers_df["fluxes"].abs()
    #Remove reactions with zero flux3_P_SERINE_
    consumers_df = consumers_df[(consumers_df.T != 0).all()]

    producers_df["Status"] = "Producer"
    consumers_df["Status"] = "Consumer"

    frame = [producers_df, consumers_df]

    all_reactions = pd.concat(frame)

    all_reactions["label"] = all_reactions.index

    #Export full dataframe to csv
    #all_reactions.to_csv("budget_plot.csv", header=True)

    #Correct Budget stoichiometry
    atp_stoi=[]
    for i in all_reactions['label']:
        for met in budget_metabolites:
            try:
                rxn=c4_model.reactions.get_by_id(i).get_coefficient(met)
                atp_stoi.append(abs(rxn))
            except KeyError:
                continue

    new_flux = all_reactions["fluxes"] * atp_stoi

    all_reactions.insert(3,'coefficient', atp_stoi)
    all_reactions.insert(4,'new_flux', new_flux)

    return producers_df

In [12]:
# Create dictionaries to store results
budget_results = {}
all_co2_producers = set()
recurring_co2_producers = {}
universal_co2_producers = set()  # For reactions that appear in ALL flux solutions

CO2_List = ["CARBON_DIOXIDE_c", "CARBON_DIOXIDE_m", "CARBON_DIOXIDE_p"]

# Track which reactions appear in which columns
reaction_to_columns = {}

def is_transporter(reaction_id):
    """Check if a reaction is a transporter (ends with mc or pc)"""
    return reaction_id.endswith('_mc') or reaction_id.endswith('_pc')

for column_name in filtered_df.columns:
    column_co2_producers = set()
    
    for CO2_metabolite in CO2_List:
        # Convert the column to a DataFrame with 'fluxes' column
        column_data = filtered_df[column_name]
        solution_frame = pd.DataFrame({
            'fluxes': column_data
        })
        
        # Run budget_plot for this column and CO2 metabolite
        try:
            producers_df = budget_plot(
                solution_frame=solution_frame,
                metabolite_query=CO2_metabolite,
                cell_type="[B]"
            )
            
            # Get the unique reaction IDs from this analysis
            current_producers = set(producers_df.index)
            
            # FILTER OUT TRANSPORTERS (mc and pc)
            current_producers = {rxn for rxn in current_producers if not is_transporter(rxn)}
            
            # Add to the global set of all unique CO2 producers
            all_co2_producers.update(current_producers)
            
            # Add to this column's producers
            column_co2_producers.update(current_producers)
            
            # Update frequency count for recurring producers
            for reaction_id in current_producers:
                recurring_co2_producers[reaction_id] = recurring_co2_producers.get(reaction_id, 0) + 1
                
                # Track which columns each reaction appears in
                if reaction_id not in reaction_to_columns:
                    reaction_to_columns[reaction_id] = set()
                reaction_to_columns[reaction_id].add(column_name)
            
            #print(f"✓ {column_name} - {CO2_metabolite}: Found {len(current_producers)} producers (after filtering transporters)")
            
        except Exception as e:
            #print(f"✗ {column_name} - {CO2_metabolite}: {e}")
            pass
    
    # Store results for this column
    budget_results[column_name] = column_co2_producers

# Identify universal producers (appear in ALL flux solutions)
total_columns = len(filtered_df.columns)
universal_co2_producers = set()

for reaction_id, columns_set in reaction_to_columns.items():
    if len(columns_set) == total_columns:
        universal_co2_producers.add(reaction_id)

# SCENARIO 1: All unique reactions that produce CO2
print("\n" + "="*80)
print("SCENARIO 1: ALL UNIQUE CO2 PRODUCTION REACTIONS")
print("="*80)
print(f"Total unique CO2-producing reactions (excluding transporters): {len(all_co2_producers)}")

if all_co2_producers:
    print("\nUnique CO2 production reactions:")
    for i, reaction_id in enumerate(sorted(all_co2_producers), 1):
        frequency = recurring_co2_producers.get(reaction_id, 0)
        print(f"{i:3d}. {reaction_id}")

# SCENARIO 2: Recurring CO2 production reactions
print("\n" + "="*80)
print("SCENARIO 2: RECURRING CO2 PRODUCTION REACTIONS")
print("="*80)
min_occurrences = 2
recurring_reactions = {reaction: count for reaction, count in recurring_co2_producers.items() 
                      if count >= min_occurrences}
recurring_sorted = sorted(recurring_reactions.items(), key=lambda x: x[1], reverse=True)

print(f"Recurring CO2 production reactions (appear in ≥{min_occurrences} flux solutions, excluding transporters): {len(recurring_sorted)}")

# NEW: UNIVERSAL CO2 PRODUCTION REACTIONS
print("\n" + "="*80)
print("SCENARIO 3 : UNIVERSAL CO2 PRODUCTION REACTIONS")
print("="*80)
print(f"Universal CO2 producers (appear in ALL {total_columns} flux solutions, excluding transporters): {len(universal_co2_producers)}")

if universal_co2_producers:
    print("\nUniversal CO2 production reactions:")
    for i, reaction_id in enumerate(sorted(universal_co2_producers), 1):
        frequency = recurring_co2_producers.get(reaction_id, 0)
        print(f"{i:3d}. {reaction_id}")
else:
    print("No universal CO2 production reactions found (after filtering transporters).")

# Additional analysis: Show reactions that appear in almost all solutions (e.g., 80%+)
print("\n" + "="*80)
print("SCENARIO 4: NEAR-UNIVERSAL CO2 PRODUCTION REACTIONS (≥70% of solutions)")
print("="*80)

min_percentage = 0.7  # 70%
min_count = int(total_columns * min_percentage)

near_universal_reactions = []
for reaction_id, count in recurring_co2_producers.items():
    if count >= min_count:
        percentage = (count / total_columns) * 100
        near_universal_reactions.append((reaction_id, count, percentage))

# Sort by frequency
near_universal_reactions.sort(key=lambda x: x[1], reverse=True)

print(f"Near-universal CO2 producers (appear in ≥{min_percentage*100:.0f}% of solutions, excluding transporters): {len(near_universal_reactions)}")
for i, (reaction_id, count, percentage) in enumerate(near_universal_reactions, 1):
    print(f"{i:3d}. {reaction_id} ({count}/{total_columns} solutions, {percentage:.1f}%)")

# Breakdown of near-universal producers by compartment
print("\n" + "="*80)
print("NEAR-UNIVERSAL PRODUCERS BY COMPARTMENT")
print("="*80)

near_universal_by_compartment = {
    'cytosol': [],
    'mitochondria': [],
    'plastid': [],
    'other': []
}

# near_universal_reactions is a list of tuples: (reaction_id, count, percentage)
for reaction_id, count, percentage in near_universal_reactions:
    if reaction_id.endswith('_c'):
        near_universal_by_compartment['cytosol'].append((reaction_id, count, percentage))
    elif reaction_id.endswith('_m'):
        near_universal_by_compartment['mitochondria'].append((reaction_id, count, percentage))
    elif reaction_id.endswith('_p'):
        near_universal_by_compartment['plastid'].append((reaction_id, count, percentage))
    else:
        near_universal_by_compartment['other'].append((reaction_id, count, percentage))

for compartment, reactions in near_universal_by_compartment.items():
    print(f"\n{compartment.upper()}: {len(reactions)} near-universal reactions")
    for i, (reaction, count, pct) in enumerate(reactions, 1):
        print(f"  {i}. {reaction} ({count}/{total_columns}, {pct:.1f}%)")

# Save results
import json

results_to_save = {
    'all_unique_reactions': list(all_co2_producers),
    'universal_reactions': list(universal_co2_producers),
    'near_universal_reactions': near_universal_reactions,
    'recurring_reactions': recurring_co2_producers,
    'compartment_breakdown': near_universal_by_compartment,
    'total_flux_solutions': total_columns,
    'note': 'Transporters (mc and pc) have been filtered out'
}


SCENARIO 1: ALL UNIQUE CO2 PRODUCTION REACTIONS
Total unique CO2-producing reactions (excluding transporters): 21

Unique CO2 production reactions:
  1. [B]_1_PERIOD_1_PERIOD_1_PERIOD_39_RXN_m
  2. [B]_2_PERIOD_3_PERIOD_1_PERIOD_180_RXN_p
  3. [B]_ACETOLACTSYN_RXN_p
  4. [B]_ACETOOHBUTSYN_RXN_p
  5. [B]_CARBOXYCYCLOHEXADIENYL_DEHYDRATASE_RXN_p
  6. [B]_DIAMINOPIMDECARB_RXN_p
  7. [B]_GCVMULTI_RXN_m
  8. [B]_IGPSYN_RXN_p
  9. [B]_ISOCITDEH_RXN_c
 10. [B]_MALIC_NADP_RXN_p
 11. [B]_PEPCARBOXYKIN_RXN_c
 12. [B]_PYRUVDEH_RXN_m
 13. [B]_PYRUVDEH_RXN_p
 14. [B]_RXN_5682_p
 15. [B]_RXN_7800_p
 16. [B]_RXN_9648_p
 17. [B]_RXN_9650_p
 18. [B]_RXN_9651_p
 19. [B]_RXN_9652_p
 20. [B]_RXN_9653_p
 21. [B]_RXN_9654_p

SCENARIO 2: RECURRING CO2 PRODUCTION REACTIONS
Recurring CO2 production reactions (appear in ≥2 flux solutions, excluding transporters): 21

SCENARIO 3 : UNIVERSAL CO2 PRODUCTION REACTIONS
Universal CO2 producers (appear in ALL 40 flux solutions, excluding transporters): 3

Universal C

In [13]:
near_universal_reactions

[('[B]_ISOCITDEH_RXN_c', 40, 100.0),
 ('[B]_RXN_7800_p', 40, 100.0),
 ('[B]_ACETOLACTSYN_RXN_p', 40, 100.0),
 ('[B]_PYRUVDEH_RXN_m', 37, 92.5)]

In [14]:
# Track aggregate fluxes and column-wise contributions
reaction_total_flux = {}
column_total_co2 = {col: 0.0 for col in filtered_df.columns}
column_rxn_fluxes = {col: {} for col in filtered_df.columns}

# 1. Iterate through columns and CO2 metabolites to calculate flux values
for column_name in filtered_df.columns:
    column_data = filtered_df[column_name]
    solution_frame = pd.DataFrame({'fluxes': column_data})
    
    for CO2_metabolite in CO2_List:
        try:
            producers_df = budget_plot(
                solution_frame=solution_frame,
                metabolite_query=CO2_metabolite,
                cell_type="[B]"
            )
            
            # Access 'fluxes' column directly
            for rxn_id, row in producers_df.iterrows():
                if is_transporter(rxn_id):
                    continue
                
                # Take absolute value to represent total magnitude of CO2 production
                flux_val = abs(row['fluxes'])
                
                column_rxn_fluxes[column_name][rxn_id] = column_rxn_fluxes[column_name].get(rxn_id, 0.0) + flux_val
                column_total_co2[column_name] += flux_val
                
                reaction_total_flux[rxn_id] = reaction_total_flux.get(rxn_id, 0.0) + flux_val

        except Exception as e:
            pass

# Calculate percentage contribution 
rxn_pct_across_columns = {}

for rxn_id in reaction_total_flux.keys():
    pct_list = []
    for col in filtered_df.columns:
        tot = column_total_co2[col]
        if tot > 0:
            rxn_flux = column_rxn_fluxes[col].get(rxn_id, 0.0)
            pct_list.append((rxn_flux / tot) * 100.0)
        else:
            pct_list.append(0.0)
            
    # Filter out inactive (0.0 percentages) so statistics only consider active cases
    active_pct_list = [p for p in pct_list if p > 0]
    
    rxn_pct_across_columns[rxn_id] = {
        'mean_pct': np.mean(active_pct_list) if active_pct_list else 0.0,
        'std_pct': np.std(active_pct_list) if active_pct_list else 0.0,
        'min_pct': np.min(active_pct_list) if active_pct_list else 0.0,
        'max_pct': np.max(active_pct_list) if active_pct_list else 0.0
    }

# 3. Build summary DataFrame
contribution_df = pd.DataFrame([
    {
        'Reaction_ID': rxn_id,
        'Compartment': 'cytosol' if rxn_id.endswith('_c') else ('mitochondria' if rxn_id.endswith('_m') else ('plastid' if rxn_id.endswith('_p') else 'other')),
        'Total_Flux_Sum': reaction_total_flux[rxn_id],
        'Mean_Contribution_%': rxn_pct_across_columns[rxn_id]['mean_pct'],
        'Std_Contribution_%': rxn_pct_across_columns[rxn_id]['std_pct'],
        'Min_Contribution_%': rxn_pct_across_columns[rxn_id]['min_pct'],
        'Max_Contribution_%': rxn_pct_across_columns[rxn_id]['max_pct'],
        'Solution_Count': recurring_co2_producers.get(rxn_id, 0),
        'Solution_Frequency_%': (recurring_co2_producers.get(rxn_id, 0) / total_columns) * 100
    }
    for rxn_id in reaction_total_flux.keys()
]).sort_values(by='Solution_Frequency_%', ascending=False).reset_index(drop=True)

print("="*80)
print("QUANTITATIVE CO2 CONTRIBUTION SUMMARY (TOP PRODUCERS BY MEAN % FLUX - ACTIVE ONLY)")
print("="*80)
print(contribution_df[['Reaction_ID', 'Compartment', 'Solution_Frequency_%' , 'Mean_Contribution_%', 'Std_Contribution_%']].head(8).to_string(index=False))

print("\n" + "="*80)
print("TOTAL MEAN CO2 PRODUCTION BY COMPARTMENT (ACTIVE ONLY)")
print("="*80)
compartment_summary = contribution_df.groupby('Compartment')['Mean_Contribution_%'].sum().reset_index()
print(compartment_summary.to_string(index=False))

QUANTITATIVE CO2 CONTRIBUTION SUMMARY (TOP PRODUCERS BY MEAN % FLUX - ACTIVE ONLY)
           Reaction_ID  Compartment  Solution_Frequency_%  Mean_Contribution_%  Std_Contribution_%
   [B]_ISOCITDEH_RXN_c      cytosol                 100.0            19.650407            3.344356
[B]_ACETOLACTSYN_RXN_p      plastid                 100.0            15.010931            3.721764
        [B]_RXN_7800_p      plastid                 100.0             8.678656            1.807569
    [B]_PYRUVDEH_RXN_m mitochondria                  92.5            16.284250            5.549934
  [B]_MALIC_NADP_RXN_p      plastid                  67.5            21.231237            6.244755
    [B]_PYRUVDEH_RXN_p      plastid                  67.5            12.527157            3.792908
    [B]_GCVMULTI_RXN_m mitochondria                  62.5            14.030648            7.767218
        [B]_RXN_9651_p      plastid                  35.0             1.122771            0.242813

TOTAL MEAN CO2 PRODUCTION

In [15]:
print("\n" + "="*80)
print("Exchange reactions (≥60% of solutions)")
print("="*80)

epsilon = 0.00001 
threshold = 0.6 * len(filtered_df.columns)

active_rows_df = filtered_df[filtered_df.abs().gt(epsilon).sum(axis=1) >= threshold]

mb_active_rows = active_rows_df[active_rows_df.index.str.contains('MB]_', na=False)].copy()

mb_active_rows['Average'] = mb_active_rows.mean(axis=1)
mb_active_rows['SD'] = mb_active_rows.std(axis=1)

summary_df = mb_active_rows[['Average', 'SD']]
print(summary_df)


Exchange reactions (≥60% of solutions)
                          Average        SD
Unnamed: 0                                 
[MB]_VAL_c              -0.166874  0.080481
[MB]_NITRATE_c          -3.205423  1.030748
[MB]_WATER_c           -15.332364  2.956657
[MB]_GLT_c              -0.488934  0.063564
[MB]_MAL_c               0.269866  0.315096
[MB]_L_ASPARTATE_c       0.647560  0.288288
[MB]_SULFATE_c          -0.086293  0.024572
[MB]_OXYGEN_MOLECULE_c  -3.640063  6.015755
[MB]_LEU_c              -0.224166  0.029032


## 6. Thermodynamic Analysis using Component Contribution
Use the `equilibrator_api` to calculate the standard Gibbs free energy ($\Delta G'^\circ$) and physiological Gibbs free energy ($\Delta G'$) for the identified reactions.

In [16]:
cc = ComponentContribution()

In [17]:
def parse_compounds(compound_id):
    """
    Parse reaction ID from format '[B]_REACTION_NAME_compartment' 
    to 'REACTION-NAME' format.
    
    Args:
        reaction_id (str): Reaction ID in format '[B]_NAME_compartment'
        
    Returns:
        str: Cleaned reaction ID with underscores replaced by hyphens
    """
    # Remove '[B]_' prefix
    cleaned = compound_id.replace('[B]_', '')
    
    # Remove the last underscore and everything after it (compartment suffix)
    # Find the last underscore
    last_underscore = cleaned.rfind('_')
    if last_underscore != -1:
        cleaned = cleaned[:last_underscore]
    
    # Replace remaining underscores with hyphens
    cleaned = cleaned.replace('_', '-')
    
    return cleaned

In [18]:
def set_reaction(reaction_id):
    """
    Creates an Equilibrator Reaction object from a model reaction ID.
    Maps model metabolites to MetaCyc compound IDs.
    """
    compound_ids = []
    
    for metabolite in c4_model.reactions.get_by_id(reaction_id).metabolites:
        compound_ids.append(parse_compounds(metabolite.id))
   
    compound_dict = {cid : cc.get_compound(f"metacyc.compound:{cid}") for cid in compound_ids}

    # initialize once before the loop
    compound_dict_reaction = {}
    for metabolite in c4_model.reactions.get_by_id(reaction_id).metabolites:
        key = compound_dict[parse_compounds(metabolite.id)]
        coeff = c4_model.reactions.get_by_id(reaction_id).get_coefficient(metabolite)
        compound_dict_reaction[key] = coeff  # update instead of recreating
    
    #Print reaction details with common names
    print(", ".join([f"{c.get_common_name()}: {s}" for c, s in compound_dict_reaction.items()]))
    reaction = Reaction(compound_dict_reaction)
    print("The reaction is " + ("" if reaction.is_balanced() else "not ") + "balanced")
    return reaction

In [19]:
reaction="[B]_RXN_7800_p"
print(c4_model.reactions.get_by_id(reaction).build_reaction_string())
equi_reaction = set_reaction(reaction)
dG0_prime = cc.standard_dg_prime(equi_reaction)
print(f"ΔG'° = {dG0_prime}")
dGm_prime = cc.physiological_dg_prime(equi_reaction)
print(f"ΔG'm = {dGm_prime}")

[B]_CPD_7100_p + [B]_PROTON_p --> [B]_2K_4CH3_PENTANOATE_p + [B]_CARBON_DIOXIDE_p
H+: -1.0, 3-Carboxy-4-methyl-2-oxopentanoate: -1.0, CO2: 1.0, 4-Methyl-2-oxopentanoate: 1.0
The reaction is balanced
ΔG'° = (-14 +/- 4) kilojoule / mole
ΔG'm = (-31 +/- 4) kilojoule / mole


In [20]:
cc.p_h = Q_(8.0)
cc.p_mg = Q_(5.0)
cc.ionic_strength = Q_("0.2 M")
cc.temperature = Q_("298.15K")

In [21]:
reaction="[B]_RXN_7800_p"
print(c4_model.reactions.get_by_id(reaction).build_reaction_string())
equi_reaction = set_reaction(reaction)
dG0_prime = cc.standard_dg_prime(equi_reaction)
print(f"ΔG'° = {dG0_prime}")
dGm_prime = cc.physiological_dg_prime(equi_reaction)
print(f"ΔG'm = {dGm_prime}")

[B]_CPD_7100_p + [B]_PROTON_p --> [B]_2K_4CH3_PENTANOATE_p + [B]_CARBON_DIOXIDE_p
H+: -1.0, 3-Carboxy-4-methyl-2-oxopentanoate: -1.0, CO2: 1.0, 4-Methyl-2-oxopentanoate: 1.0
The reaction is balanced
ΔG'° = (-11 +/- 4) kilojoule / mole
ΔG'm = (-28 +/- 4) kilojoule / mole


In [22]:
# optional: changing the aqueous environment parameters
cc.p_h = Q_(7.4)
cc.p_mg = Q_(3.0)
cc.ionic_strength = Q_("0.25M")
cc.temperature = Q_("298.15K")

In [23]:
import pandas as pd
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module='equilibrator_cache')

reactions_to_check = [
    '[B]_DIAMINOPIMDECARB_RXN_p', '[B]_ACETOOHBUTSYN_RXN_p', '[B]_RXN_5682_p',
    '[B]_RXN_7800_p', '[B]_ACETOLACTSYN_RXN_p', '[B]_CARBOXYCYCLOHEXADIENYL_DEHYDRATASE_RXN_p',
    '[B]_2_PERIOD_3_PERIOD_1_PERIOD_180_RXN_p', '[B]_RXN_9648_p', '[B]_RXN_9650_p',
    '[B]_RXN_9651_p', '[B]_RXN_9652_p', '[B]_RXN_9653_p', '[B]_RXN_9654_p',
    '[B]_PYRUVDEH_RXN_p', '[B]_PYRUVDEH_RXN_m', '[B]_ISOCITDEH_RXN_c'
]

# Define environments
env_map = {
    "_c": {
        'p_h': Q_(7.2),
        'p_mg': Q_(3.0),
        'ionic_strength': Q_("0.25 M"),
        'temperature': Q_("298.15 K")
    },
    "_m": {
        'p_h': Q_(7.8),
        'p_mg': Q_(2.5),
        'ionic_strength': Q_("0.3 M"),
        'temperature': Q_("298.15 K")
    },
    "_p": {
        'p_h': Q_(8.0),
        'p_mg': Q_(5.0),
        'ionic_strength': Q_("0.2 M"),
        'temperature': Q_("298.15 K")
    }
}

def apply_environment(cc, env):
    cc.p_h = env['p_h']
    cc.p_mg = env['p_mg']
    cc.ionic_strength = env['ionic_strength']
    cc.temperature = env['temperature']

results = []

for reaction_id in reactions_to_check:
    suffix = reaction_id[-2:]
    if suffix not in env_map:
        continue

    apply_environment(cc, env_map[suffix])

    try:
        equi_reaction = set_reaction(reaction_id)

        if any(c is None for c in equi_reaction.keys()):
            results.append({"Reaction": reaction_id, "dG0_prime": "Skipped", "dGm_prime": "Skipped"})
            continue

        # Calculate values
        dg0 = cc.standard_dg_prime(equi_reaction)
        dgm = cc.physiological_dg_prime(equi_reaction)

        # Check for high uncertainty (errors > 1000) 
        if dg0.error.magnitude > 1000:
            results.append({
                "Reaction": reaction_id, 
                "dG0_prime": "n/a", 
                "dGm_prime": "n/a"
            })
        else:
            results.append({
                "Reaction": reaction_id,
                "dG0_prime": f"{dg0.value.magnitude:.2f} +/- {dg0.error.magnitude:.2f}",
                "dGm_prime": f"{dgm.value.magnitude:.2f} +/- {dgm.error.magnitude:.2f}"
            })

    except Exception as e:
        results.append({"Reaction": reaction_id, "dG0_prime": "Error", "dGm_prime": str(e)})

df = pd.DataFrame(results)
print(df.to_string(index=False))

H+: -1.0, Meso-2,6-Diaminoheptanedioate: -1.0, L-Lysine: 1.0, CO2: 1.0
The reaction is balanced
H+: -1.0, Pyruvate: -1.0, 2-Oxobutanoate: -1.0, (S)-2-Aceto-2-hydroxybutanoate: 1.0, CO2: 1.0
The reaction is balanced
Nicotinamide adenine dinucleotide phosphate: -1.0, L-Arogenate: -1.0, CO2: 1.0, Nicotinamide adenine dinucleotide phosphate - reduced: 1.0, L-Tyrosine: 1.0
The reaction is balanced
H+: -1.0, 3-Carboxy-4-methyl-2-oxopentanoate: -1.0, CO2: 1.0, 4-Methyl-2-oxopentanoate: 1.0
The reaction is balanced
H+: -1.0, Pyruvate: -2.0, (S)-2-Acetolactate: 1.0, CO2: 1.0
The reaction is balanced
L-Arogenate: -1.0, H+: -1.0, L-Phenylalanine: 1.0, H2O: 1.0, CO2: 1.0
The reaction is balanced
Malonyl-[acyl-carrier protein]: -1.0, H+: -1.0, Acetyl-CoA: -1.0, Acetoacetyl-ACP: 1.0, CO2: 1.0, Coenzyme A: 1.0
The reaction is not balanced
H+: -1.0, Butyryl-[acp]: -1.0, Malonyl-CoA: -1.0, 3-Oxohexanoyl-[acyl-carrier protein]: 1.0, CO2: 1.0, Coenzyme A: 1.0
The reaction is not balanced
Malonyl-CoA: -1.

In [24]:
import pandas as pd
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module='equilibrator_cache')

# Pull reactions directly from the summary DataFrame generated in the previous step
reactions_to_check = contribution_df['Reaction_ID'].tolist()

# Define environments
env_map = {
    "_c": {
        'p_h': Q_(7.2),
        'p_mg': Q_(3.0),
        'ionic_strength': Q_("0.25 M"),
        'temperature': Q_("298.15 K")
    },
    "_m": {
        'p_h': Q_(7.8),
        'p_mg': Q_(2.5),
        'ionic_strength': Q_("0.3 M"),
        'temperature': Q_("298.15 K")
    },
    "_p": {
        'p_h': Q_(8.0),
        'p_mg': Q_(5.0),
        'ionic_strength': Q_("0.2 M"),
        'temperature': Q_("298.15 K")
    }
}

def apply_environment(cc, env):
    cc.p_h = env['p_h']
    cc.p_mg = env['p_mg']
    cc.ionic_strength = env['ionic_strength']
    cc.temperature = env['temperature']

results = []

for reaction_id in reactions_to_check:
    suffix = reaction_id[-2:]
    if suffix not in env_map:
        continue

    apply_environment(cc, env_map[suffix])

    try:
        equi_reaction = set_reaction(reaction_id)

        if any(c is None for c in equi_reaction.keys()):
            results.append({"Reaction": reaction_id, "dG0_prime": "Skipped", "dGm_prime": "Skipped"})
            continue

        # Calculate values
        dg0 = cc.standard_dg_prime(equi_reaction)
        dgm = cc.physiological_dg_prime(equi_reaction)

        # Check for high uncertainty (errors > 1000) 
        if dg0.error.magnitude > 1000:
            results.append({
                "Reaction": reaction_id, 
                "dG0_prime": "n/a", 
                "dGm_prime": "n/a"
            })
        else:
            results.append({
                "Reaction": reaction_id,
                "dG0_prime": f"{dg0.value.magnitude:.2f} +/- {dg0.error.magnitude:.2f}",
                "dGm_prime": f"{dgm.value.magnitude:.2f} +/- {dgm.error.magnitude:.2f}"
            })

    except Exception as e:
        results.append({"Reaction": reaction_id, "dG0_prime": "Error", "dGm_prime": str(e)})

df = pd.DataFrame(results)
print(df.to_string(index=False))

Nicotinamide adenine dinucleotide phosphate: -1.0, (+)-threo-isocitric acid: -1.0, 2-Oxoglutarate: 1.0, CO2: 1.0, Nicotinamide adenine dinucleotide phosphate - reduced: 1.0
The reaction is balanced
H+: -1.0, Pyruvate: -2.0, (S)-2-Acetolactate: 1.0, CO2: 1.0
The reaction is balanced
H+: -1.0, 3-Carboxy-4-methyl-2-oxopentanoate: -1.0, CO2: 1.0, 4-Methyl-2-oxopentanoate: 1.0
The reaction is balanced
Nicotinamide adenine dinucleotide: -1.0, Coenzyme A: -1.0, Pyruvate: -1.0, CO2: 1.0, Acetyl-CoA: 1.0, Nicotinamide adenine dinucleotide - reduced: 1.0
The reaction is balanced
L-Malate: -1.0, Nicotinamide adenine dinucleotide phosphate: -1.0, Pyruvate: 1.0, CO2: 1.0, Nicotinamide adenine dinucleotide phosphate - reduced: 1.0
The reaction is balanced
Pyruvate: -1.0, Nicotinamide adenine dinucleotide: -1.0, Coenzyme A: -1.0, Nicotinamide adenine dinucleotide - reduced: 1.0, Acetyl-CoA: 1.0, CO2: 1.0
The reaction is balanced
5,6,7,8-Tetrahydrofolate: -1.0, Glycine: -1.0, Nicotinamide adenine dinu